

# Autonomous Fraud Detection & Response Network

Stage-2 POC Implementation

Name: Meerala Nithisha
                                       College: St.Martins Engineering College

## Project Objective

The goal of this project is to develop an intelligent system capable of identifying fraudulent insurance claims.

Key objectives:
• Detect fraudulent claims using machine learning  
• Handle imbalanced fraud datasets  
• Generate fraud probability scores  
• Provide automated response decisions for claim processing

## 1. Import Required Libraries

In this step, the required Python libraries are imported.

Pandas and NumPy are used for data manipulation and analysis.

In [ ]:
import pandas as pd
import numpy as np

## 2. Load Dataset

The insurance fraud dataset is loaded using the Pandas library.

Each row represents an insurance claim and each column represents a feature describing the claim.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/insurance fraud claims.csv')
df.head()

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,...,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,...,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,...,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,...,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,NaN
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,...,NO,6500,1300,650,4550,Accura,RSX,2009,N,NaN


## 3. Dataset Information

This step provides information about the dataset structure including the number of rows, columns, and data types.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           1000 non-null   int64  
 1   age                          1000 non-null   int64  
 2   policy_number                1000 non-null   int64  
 3   policy_bind_date             1000 non-null   object 
 4   policy_state                 1000 non-null   object 
 5   policy_csl                   1000 non-null   object 
 6   policy_deductable            1000 non-null   int64  
 7   policy_annual_premium        1000 non-null   float64
 8   umbrella_limit               1000 non-null   int64  
 9   insured_zip                  1000 non-null   int64  
 10  insured_sex                  1000 non-null   object 
 11  insured_education_level      1000 non-null   object 
 12  insured_occupation           1000 non-null   object 
 13  insured_hobbies    

## 4. Data Preprocessing

Data preprocessing prepares the dataset for machine learning.

Steps performed:
• Remove unnecessary columns  
• Handle missing values  
• Encode categorical variables  
• Convert fraud labels into numeric format

### Removing unnecessary columns
These columns do not contribute significantly to fraud prediction.

In [ ]:
df.drop(["policy_number","policy_bind_date","incident_date"], axis=1, inplace=True)
df.drop("_c39", axis=1, inplace=True)

### Handling Missing Values

Missing values in the authorities_contacted column are replaced with "Unknown".


In [ ]:
df["authorities_contacted"] = df["authorities_contacted"].fillna("Unknown")

### Convert Fraud Labels

The fraud_reported column is converted into binary format:

1 → Fraud  
0 → Genuine Claim

In [ ]:
df["fraud_reported"] = df["fraud_reported"].map({"Y":1,"N":0})

### Encoding Categorical Variables

Machine learning models require numerical input.  
Categorical text values are converted into numeric format using Label Encoding.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for column in df.columns:
    if df[column].dtype == "object":
        df[column] = le.fit_transform(df[column])

## 5. Feature Preparation

The dataset is divided into:

X → Input features  
y → Target variable (fraud label)

In [ ]:
X = df.drop("fraud_reported", axis=1)
y = df["fraud_reported"]

## 6. Train-Test Split

The dataset is divided into training and testing sets.

Training data → 80%  
Testing data → 20%

The training dataset is used to train the model while the testing dataset evaluates model performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 7. Handling Class Imbalance using SMOTE

Fraud datasets usually contain fewer fraud cases than genuine cases.  
SMOTE (Synthetic Minority Over-sampling Technique) is used to generate synthetic fraud samples to balance the dataset.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

## 8. Train Fraud Detection Model

A Random Forest classifier is used for fraud detection because it performs well with structured tabular datasets and reduces overfitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42
)

model.fit(X_train_smote, y_train_smote)

RandomForestClassifier(max_depth=12, n_estimators=300, random_state=42)

## 9. Model Prediction

The trained model predicts whether each claim in the test dataset is fraudulent or genuine.

In [ ]:
y_pred = model.predict(X_test)

## 10. Model Evaluation

Model performance is evaluated using accuracy, precision, recall and F1-score.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.74
              precision    recall  f1-score   support

           0       0.81      0.83      0.82       145
           1       0.53      0.49      0.51        55

    accuracy                           0.74       200
   macro avg       0.67      0.66      0.67       200
weighted avg       0.73      0.74      0.74       200





The Random Forest model achieved an accuracy of approximately 74% on the test dataset.

The model performs well in identifying genuine insurance claims while detecting a significant portion of fraudulent cases.

Fraud detection recall is slightly lower due to the class imbalance in the dataset. To address this issue, SMOTE was applied to balance the training data.

Future improvements may include advanced models such as Gradient Boosting or XGBoost to further improve fraud detection performance.

## 11. Fraud Risk Scoring

Instead of only predicting fraud, the model generates a fraud probability score representing the risk level of the claim.

In [ ]:
fraud_probability = model.predict_proba(X_test)[:,1]
fraud_probability[:10]

array([0.16258734, 0.12379795, 0.22652161, 0.24334401, 0.11752438,
       0.31828297, 0.51827019, 0.36550186, 0.13041842, 0.11059775])

## 12. Autonomous Fraud Response

Based on fraud probability, the system automatically determines the action.

In [ ]:
def fraud_decision(prob):

    if prob > 0.8:
        return "High Risk Fraud – Investigation Required"

    elif prob > 0.5:
        return "Medium Risk – Manual Review"

    else:
        return "Low Risk – Approve Claim"

### Testing Autonomous Decision System

In [ ]:
for i in range(5):

    print("Fraud Probability:", fraud_probability[i])
    print("Decision:", fraud_decision(fraud_probability[i]))
    print("-----------")

Fraud Probability: 0.16258734361837668
Decision: Low Risk – Approve Claim
-----------
Fraud Probability: 0.12379794520547945
Decision: Low Risk – Approve Claim
-----------
Fraud Probability: 0.2265216149537847
Decision: Low Risk – Approve Claim
-----------
Fraud Probability: 0.24334401440944306
Decision: Low Risk – Approve Claim
-----------
Fraud Probability: 0.11752437864746272
Decision: Low Risk – Approve Claim
-----------


## Agent-Based Architecture

The system is designed as a multi-agent architecture:

• Data Ingestion Agent – collects claim data  
• Preprocessing Agent – cleans and prepares data  
• Fraud Detection Agent – predicts fraud using ML model  
• Risk Scoring Agent – generates fraud probability  
• Decision Agent – provides automated response  

This forms an Autonomous Fraud Detection and Response Network.


## Final System Pipeline

Insurance Claim Data  
↓  
Data Preprocessing  
↓  
Feature Engineering  
↓  
SMOTE Balancing  
↓  
Random Forest Fraud Detection Model  
↓  
Fraud Prediction  
↓  
Risk Scoring  
↓  
Autonomous Decision System

## Conclusion

In this notebook, we implemented a machine learning-based fraud detection system for insurance claims.

Key contributions:

• Performed data preprocessing and feature engineering  
• Handled class imbalance using SMOTE  
• Trained a Random Forest model for fraud detection  
• Evaluated model performance using classification metrics  
• Generated fraud risk scores and automated decision logic

This system demonstrates how machine learning can assist in identifying fraudulent insurance claims and improving decision-making processes.